In [1]:
import json

from glob import glob
from unidiff import PatchSet

In [2]:
!git clone git@github.com:CognitionAI/devin-swebench-results.git

fatal: destination path 'devin-swebench-results' already exists and is not an empty directory.


In [3]:
def convert_devin_txt_to_pred(pred_file):
    inst_id = pred_file.split("/")[-1].split("-diff")[0]
    pred = open(pred_file).read()
    try:
        PatchSet(pred)
    except:
        print(f"{inst_id}: Prediction patch is malformed")
    return {
        "model_name_or_path": "devin-20240406",
        "instance_id": inst_id,
        "model_patch": pred
    }

In [4]:
predictions = []
for pred_file in \
    glob("devin-swebench-results/output_diffs/fail/*.txt") + \
    glob("devin-swebench-results/output_diffs/pass/*.txt"):
    predictions.append(convert_devin_txt_to_pred(pred_file))
len(predictions)

570

In [5]:
predictions[0]

{'model_name_or_path': 'devin-20240406',
 'instance_id': 'pydata__xarray-3239',
 'model_patch': 'diff --git a/xarray/backends/api.py b/xarray/backends/api.py\nindex a20d3c2a..f476eafa 100644\n--- a/xarray/backends/api.py\n+++ b/xarray/backends/api.py\n@@ -486,9 +486,10 @@ def open_dataset(\n     if isinstance(filename_or_obj, Path):\n         filename_or_obj = str(filename_or_obj)\n \n+    store = None\n+\n     if isinstance(filename_or_obj, AbstractDataStore):\n         store = filename_or_obj\n-\n     elif isinstance(filename_or_obj, str):\n         filename_or_obj = _normalize_path(filename_or_obj)\n \n@@ -516,7 +517,6 @@ def open_dataset(\n             store = backends.CfGribDataStore(\n                 filename_or_obj, lock=lock, **backend_kwargs\n             )\n-\n     else:\n         if engine not in [None, "scipy", "h5netcdf"]:\n             raise ValueError(\n@@ -531,6 +531,9 @@ def open_dataset(\n                 filename_or_obj, group=group, lock=lock, **backend_kwargs\n   

In [6]:
with open("devin_predictions.jsonl", "w") as f:
    for pred in predictions:
        print(json.dumps(pred), file=f, flush=True)